#พื้นฐาน Python ที่จำเป็น (Essential Python Basics)

**อยู่ก่อนหัวข้อ "ภาพรวม Image Processing" ในสไลด์** (สไลด์ 12-17) — พื้นฐาน Python เท่าที่จำเป็น
สำหรับเขียนโค้ดประมวลผลภาพในบทถัดไป

หัวข้อย่อย : 1) ตัวแปร/ชนิดข้อมูล 2) Boolean/Set/Dict 3) List/NumPy/Tuple/String
4) for/while Loop 5) Function


> เนื้อหาและภาพตัวอย่างอ้างอิงจากสไลด์ **COMPUTER VISIONS** โดย Asst.Prof.Dr. Amnach Khawne, King Mongkut's Institute of Technology Ladkrabang


## วิธีใช้ Notebook นี้

- รันทีละ Cell จากบนลงล่างด้วย **Shift + Enter**
- แต่ละหัวข้อแบ่งเป็น Cell ย่อยหลาย Cell ทำทีละขั้นตอน เพื่อให้เห็นผลลัพธ์ทันทีทีละ Cell
- ลองแก้ค่าตัวเลข (parameter) แล้วรันซ้ำ เพื่อดูว่าภาพเปลี่ยนไปอย่างไร
- **ต้องอัปโหลด `data.zip` ก่อน** (Cell ที่ 2) ทุกครั้งที่เปิด Notebook ใหม่ — Colab ลบไฟล์ทิ้งเมื่อ Runtime ถูกรีเซ็ต


## 0. เตรียมเครื่องมือ (Setup)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow  # แสดงภาพใน Colab แทน cv2.imshow()

print("OpenCV:", cv2.__version__)
print("NumPy :", np.__version__)


In [ ]:
def show_images(images, titles, cmap=None, figsize=(15, 5)):
    """แสดงภาพหลายภาพเรียงกันในแถวเดียว สำหรับเปรียบเทียบก่อน-หลัง"""
    n = len(images)
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(1, n, i + 1)
        if img.ndim == 2:
            plt.imshow(img, cmap=cmap or "gray", vmin=0, vmax=255)
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))  # BGR -> RGB สำหรับ matplotlib
        plt.title(title, fontsize=11)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


## 1. เตรียมชุดข้อมูล (Dataset) — อัปโหลด `data.zip`

อัปโหลดไฟล์ **`data.zip`** ที่ได้รับจากผู้สอน (ภาพชุดเดียวกับที่ใช้ในสไลด์ COMPUTER VISIONS)
เมื่อรัน Cell ด้านล่างจะมีปุ่มให้เลือกไฟล์จากเครื่อง — เลือก `data.zip` แล้วรอจนแตกไฟล์เสร็จ

In [ ]:
import os
import zipfile

from google.colab import files

%cd /content
print("เลือกไฟล์ data.zip (ชุดภาพตัวอย่างจากสไลด์ COMPUTER VISIONS)")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall("/content/")

DATA_DIR = "/content/data"
print(f"แตกไฟล์ {zip_filename} เรียบร้อยแล้ว")
print("ไฟล์ภาพที่มีในโฟลเดอร์ data/:")
for fname in sorted(os.listdir(DATA_DIR)):
    print(" -", fname)


## 1. ตัวแปร (Variable) และชนิดข้อมูล — `int`, `float` (สไลด์ 13)

- **`int`** (จำนวนเต็ม) — เช่น ค่าความเข้มพิกเซลของภาพระดับเทา 8 บิต (0 = ดำ, 255 = ขาว)
- **`float`** (จำนวนทศนิยม) — เช่น ตัวคูณปรับความสว่าง (< 1 มืดลง, > 1 สว่างขึ้น)

In [ ]:
pixel_value = 180          # int
brightness_factor = 0.5    # float

print(type(pixel_value), type(brightness_factor))


In [ ]:
adjusted = pixel_value * brightness_factor
print("pixel_value * brightness_factor =", adjusted, "->", type(adjusted))


## 2. Boolean, Set และ Dictionary (สไลด์ 14)

- **`bool`** — จริง/เท็จ เช่น ใช้ตัดสินว่าพิกเซลนี้ "เป็นวัตถุ" หรือ "เป็นพื้นหลัง"
- **`dict`** — จับคู่ Key → Value เช่น เก็บค่า threshold ของแต่ละไฟล์ภาพ
- **`set`** — เก็บค่าที่ไม่ซ้ำ

In [ ]:
pixel_value = 180
is_foreground = pixel_value > 127
print("is_foreground =", is_foreground)


In [ ]:
thresholds = {"document.png": 180, "coins.png": 127}
print("threshold ของ coins.png:", thresholds["coins.png"])


In [ ]:
levels = {50, 120, 120, 200, 50}   # ค่าซ้ำถูกตัดออกอัตโนมัติ
print("levels:", levels)
print("levels เรียงลำดับ:", sorted(levels))


## 3. List, NumPy Array, Tuple และ String

| ชนิดข้อมูล | ใช้เก็บอะไร | แก้ไขได้ไหม |
|---|---|---|
| **List** | รายชื่อไฟล์ภาพ | ✅ |
| **NumPy ndarray** | ข้อมูลพิกเซลของภาพ (H×W×C) | ✅ |
| **Tuple** | มิติของภาพ (`image.shape`) | ❌ |
| **String** | ชื่อไฟล์/พาธ | ❌ |

In [ ]:
image_files = ["pcb.jpg", "flower.jpg", "oranges.jpg"]
image_files.append("label_tilted.jpg")
print(image_files)
print("ไฟล์แรก:", image_files[0])


In [ ]:
image = cv2.imread(f"{DATA_DIR}/pcb.jpg")

print("type :", type(image))
print("shape:", image.shape, "(H, W, C)")
print("dtype:", image.dtype)
print("image[0, 0]:", image[0, 0], "(พิกเซลมุมบนซ้าย เป็น [B, G, R])")


In [ ]:
shape = image.shape
print("shape =", shape, type(shape))

try:
    shape[0] = 999
except TypeError as e:
    print("แก้ไขค่าใน tuple ไม่ได้:", e)


In [ ]:
filename = "images/pcb.jpg"

folder, name = filename.split("/")
base, ext = name.split(".")
new_filename = f"{base}_gray.{ext}"

print("โฟลเดอร์:", folder, "| ชื่อ:", base, "| นามสกุล:", ext)
print("ชื่อไฟล์ใหม่:", new_filename)


## 4. for Loop vs while Loop (สไลด์ 16)

- **for loop** — วนทีละสมาชิกจนครบ รู้จำนวนรายการล่วงหน้า (เช่น ไฟล์ภาพทั้งหมดในโฟลเดอร์)
- **while loop** — ทำซ้ำขณะเงื่อนไขยังจริง ไม่รู้จำนวนรอบล่วงหน้า (เช่น อ่านเฟรมกล้องไปเรื่อย ๆ)

In [ ]:
image_files = ["pcb.jpg", "flower.jpg", "oranges.jpg", "label_tilted.jpg"]

for fname in image_files:
    print("กำลังประมวลผล:", fname)


In [ ]:
frame_count = 0
max_frames = 5

while frame_count < max_frames:
    print("อ่านเฟรมที่:", frame_count)
    frame_count += 1   # ต้องอัปเดตเอง ไม่เช่นนั้นจะวนไม่รู้จบ


## 5. Function

`def ชื่อฟังก์ชัน(parameter):` → ทำงาน → `return` ส่งผลลัพธ์กลับ
- **parameter** = ตัวแปรรับข้อมูลตอนประกาศฟังก์ชัน
- **argument** = ค่าจริงที่ส่งเข้าไปตอนเรียกใช้

In [ ]:
def to_grayscale(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

frame = cv2.imread(f"{DATA_DIR}/pcb.jpg")
gray_image = to_grayscale(frame)

print("input shape :", frame.shape)
print("output shape:", gray_image.shape)


In [ ]:
show_images([frame, gray_image], ["Input: frame", "to_grayscale(frame)"], cmap="gray")


In [ ]:
def adjust_brightness(image, value):   # 2 parameters
    return cv2.add(image, np.full(image.shape, value, dtype=np.uint8))

bright_result = adjust_brightness(frame, 60)   # 2 arguments
show_images([frame, bright_result], ["Input: frame", "adjust_brightness(frame, 60)"])


## สรุปสิ่งที่เรียนในบทนี้

| แนวคิด | ตัวอย่างที่ใช้ | ทำไมสำคัญกับงานภาพ |
|---|---|---|
| `int` / `float` | ค่าพิกเซล, ตัวคูณความสว่าง | ค่าพิกเซลเป็น int (0-255), การคำนวณปรับภาพมักได้ float |
| `bool` | `pixel_value > 127` | ใช้ตัดสิน Pass/Fail, Foreground/Background |
| `dict` | `thresholds["coins.png"]` | เก็บพารามิเตอร์ต่อภาพ/ต่องานได้เป็นระเบียบ |
| `set` | `sorted(levels)` | หาค่าไม่ซ้ำ เช่น labels ในภาพ segmentation |
| `list` | `image_files` | เก็บรายชื่อไฟล์ที่ต้องวนประมวลผล |
| `numpy.ndarray` | `image.shape`, `image[y, x]` | โครงสร้างข้อมูลหลักของภาพใน OpenCV |
| `tuple` | `image.shape` | มิติภาพต้องไม่ถูกแก้โดยไม่ตั้งใจ |
| `str` | ชื่อไฟล์ภาพ | สร้าง/แยกชื่อไฟล์ผลลัพธ์ |
| `for` / `while` | วนไฟล์ / วนอ่านเฟรม | ประมวลผลภาพจำนวนมากโดยไม่เขียนซ้ำ |
| `def` | `to_grayscale()`, `adjust_brightness()` | จัดโค้ดเป็นส่วน ๆ นำกลับมาใช้ซ้ำได้ |

**ถัดไป:เข้าใจข้อมูลภาพ (Pixel / Resolution / Channel / dtype)


## แบบฝึกหัดท้ายบท (ลองทำเอง)

1. เปลี่ยนค่า `pixel_value` ในหัวข้อ 2 เป็น 100 แล้วดู `is_foreground`
2. เพิ่มไฟล์ใหม่ลงใน `thresholds` (dict) เช่น `"screw.png": 140`
3. แก้ `max_frames` ในหัวข้อ 4 เป็น 10
4. เขียนฟังก์ชันใหม่ `adjust_contrast(image, alpha)` โดยใช้ `cv2.convertScaleAbs(image, alpha=alpha, beta=0)` แล้วเรียกใช้กับ `frame`
